In [ ]:
!pip install behave sentence-transformers pandas

In [1]:
from google.colab import files
uploaded = files.upload()


Saving pilot_llm_output.csv to pilot_llm_output (1).csv


In [4]:
import pandas as pd
import re
import subprocess
import os
import shutil
import numpy as np
from sentence_transformers import SentenceTransformer, util
from IPython.display import display

# ==========================================
# PHẦN 1: KHỞI TẠO VÀ CẤU HÌNH
# ==========================================
print("1. Đang tải model NLP (SentenceTransformers)...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# Dữ liệu Ground Truth
mock_ground_truth = {
    "US002": "Feature: Transaction History\n  Scenario: Check transaction history and keep a record\n    Given the user is logged into the system\n    When the user accesses the transaction history section\n    Then the user should see a detailed record of all past transactions",
    "US006": "Feature: OpenRefine Reconciliation API\n  Scenario: Generate endpoint from Tabular Data Package\n    Given the researcher has a Tabular Data Package\n    When they upload the package to the app\n    Then the app should generate an OpenRefine reconciliation API endpoint for data cleaning",
    "US007": "Feature: DSpace Integration\n  Scenario: Integrate data-packaged data with DSpace pipelines\n    Given the developer has data-packaged data\n    When they trigger the DSpace integration process\n    Then the data should be successfully integrated into the DSpace pipeline",
    "US008": "Feature: Environment-friendly Facilities\n  Scenario: Filter facilities by eco-friendly status\n    Given the user is browsing the list of facilities\n    When they apply the environment-friendly filter\n    Then the system should only display facilities that do not leave a negative ecological footprint",
    "US009": "Feature: Project DMP Linking\n  Scenario: Link datasets with project DMP\n    Given the depositor has a dataset ready for submission\n    When they link the dataset to the corresponding project DMP\n    Then the system should demonstrate compliance and link the whole project workflow together",
    "US015": "Feature: Building Development Project Completion\n  Scenario: Complete project to obtain occupancy\n    Given the Applicant has finished all building phases\n    When they submit the completion request for the Building Development Project\n    Then the system should process the release of any posted bond\n    And grant the occupancy permit",
    "US016": "Feature: Training Options Display\n  Scenario: View comprehensive training details on a single page\n    Given the anonymous user is on the training page\n    When they browse the available training options\n    Then they should see clearly marked details including day, time, title, summary, trainers, level, registrations, location, and a register link on one page",
    "US018": "Feature: Submission Creator Tracking\n  Scenario: View who created a submission\n    Given the Agency user is viewing a specific submission\n    When they check the submission details\n    Then the system should accurately display the name of the user who created the submission",
    "US035": "Feature: Course and Event Management\n  Scenario: Copy an existing course\n    Given the trainer has an existing course or event\n    When they select the option to copy it\n    Then the system should create a new draft course with identical details",
    "US041": "Feature: Visualisation Textual Descriptions\n  Scenario: View descriptions accompanying visualisations\n    Given the Data Consuming User is viewing an embedded visualisation\n    When the visualisation is rendered on the screen\n    Then a relevant textual description should accompany it to aid understanding"
}

# ==========================================
# PHẦN 2: CÁC HÀM XỬ LÝ LÕI
# ==========================================
# ==========================================
# PHẦN 2: CÁC HÀM XỬ LÝ LÕI (ĐÃ TỐI ƯU CHO RQ2)
# ==========================================
from behave.parser import parse_feature, ParserError

def extract_gherkin(text):
    """Trích xuất mã Gherkin từ output của LLM"""
    if pd.isna(text): return ""
    text_str = str(text)

    match = re.search(r'\x60\x60\x60(?:gherkin|feature)?\n(.*?)\n\x60\x60\x60', text_str, re.IGNORECASE | re.DOTALL)
    if match:
        extracted = match.group(1).strip()
        if 'Feature:' in extracted:
            return extracted

    if 'Feature:' in text_str:
        lines = text_str.split('\n')
        gherkin_lines = []
        is_gherkin = False
        for line in lines:
            if line.strip().startswith('Feature:'):
                is_gherkin = True
            if is_gherkin:
                if line.strip().startswith('\x60\x60\x60') or line.strip().startswith('#'):
                    break
                gherkin_lines.append(line)
        return '\n'.join(gherkin_lines).strip()
    return ""

def check_syntax_validity(gherkin_text):
    """
    Kiểm tra độ hợp lệ cú pháp (Syntax) bằng Behave Parser thuần túy.
    Bỏ qua lỗi Undefined Steps của code Python.
    """
    if not gherkin_text or 'Feature:' not in gherkin_text:
        return 0, "Lỗi: Code rỗng hoặc thiếu từ khóa 'Feature:'"

    try:
        # Dùng trực tiếp lõi phân tích ngữ pháp của behave
        parse_feature(gherkin_text)
        return 1, "Hợp lệ (Cú pháp chuẩn xác)"
    except ParserError as e:
        # Nếu bắt được ParserError, chắc chắn 100% LLM viết sai syntax
        error_line = str(e).strip().split('\n')[0]
        return 0, f"Sai cú pháp Gherkin: {error_line}"
    except Exception as e:
        return 0, f"Lỗi khác: {str(e)}"

    print("✅ Đã cập nhật hàm Parser thông minh cho RQ2!")

    # File cấu hình bắt buộc cho behave
    with open(os.path.join(steps_dir, 'dummy_steps.py'), 'w', encoding='utf-8') as f:
        f.write("from behave import *\n# Dummy file\n")

    feature_file = os.path.join(features_dir, 'test_scenario.feature')
    with open(feature_file, 'w', encoding='utf-8') as f:
        f.write(gherkin_text)

    try:
        result = subprocess.run(
            ['behave', '--dry-run', '--no-capture', feature_file],
            capture_output=True,
            text=True
        )
        if result.returncode == 0:
            return 1, "Hợp lệ"
        else:
            error_msg = result.stderr.strip() if result.stderr else result.stdout.strip()
            short_error = error_msg[:200] + "..." if len(error_msg) > 200 else error_msg
            return 0, short_error
    except Exception as e:
        return 0, f"Lỗi hệ thống: {str(e)}"

# ==========================================
# PHẦN 3: ĐƯỜNG ỐNG THỰC THI (PIPELINE)
# ==========================================
def run_full_pipeline(csv_path):
    print(f"2. Đang đọc dữ liệu từ file {csv_path}...")
    try:
        df = pd.read_csv(csv_path)
    except FileNotFoundError:
        print("LỖI: Không tìm thấy file csv. Hãy đảm bảo bạn đã tải file lên Colab!")
        return None

    # Trích xuất
    print("3. Đang trích xuất mã Gherkin từ nội dung LLM...")
    df['extracted_gherkin'] = df['llm_output'].apply(extract_gherkin)

    # Chạy RQ2
    print("4. Đang kiểm tra cú pháp (RQ2)...")
    rq2_results = df['extracted_gherkin'].apply(check_syntax_validity)

    # Tách tuple (điểm, thông báo) thành 2 cột riêng biệt một cách an toàn
    df['rq2_validity'] = [res[0] for res in rq2_results]
    df['rq2_message'] = [res[1] for res in rq2_results]

    # Chạy RQ1
    print("5. Đang đo lường độ tương đồng ngữ nghĩa (RQ1)...")
    semantic_scores = []
    for index, row in df.iterrows():
        story_id = row['story_id']
        llm_gherkin = row['extracted_gherkin']
        ground_truth = mock_ground_truth.get(story_id, "")

        if llm_gherkin and ground_truth:
            emb1 = model.encode(llm_gherkin, convert_to_tensor=True)
            emb2 = model.encode(ground_truth, convert_to_tensor=True)
            semantic_scores.append(util.cos_sim(emb1, emb2).item())
        else:
            semantic_scores.append(np.nan)

    df['rq1_similarity'] = semantic_scores

    return df

# ==========================================
# KÍCH HOẠT CHẠY
# ==========================================
# Lưu ý: File dữ liệu của bạn phải có tên chính xác là 'pilot_llm_output.csv'
final_df = run_full_pipeline('pilot_llm_output.csv')

if final_df is not None:
    print("\n" + "="*40)
    print("HOÀN THÀNH! DƯỚI ĐÂY LÀ KẾT QUẢ ĐO LƯỜNG:")
    print("="*40)

    # Hiển thị bảng đẹp trên Colab
    display(final_df[['story_id', 'rq2_validity', 'rq2_message', 'rq1_similarity']])

    # Xuất file
    output_filename = 'final_measurement_results.csv'
    final_df.to_csv(output_filename, index=False)
    print(f"\nĐã xuất file kết quả: {output_filename}")

1. Đang tải model NLP (SentenceTransformers)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

2. Đang đọc dữ liệu từ file pilot_llm_output.csv...
3. Đang trích xuất mã Gherkin từ nội dung LLM...
4. Đang kiểm tra cú pháp (RQ2)...
5. Đang đo lường độ tương đồng ngữ nghĩa (RQ1)...

HOÀN THÀNH! DƯỚI ĐÂY LÀ KẾT QUẢ ĐO LƯỜNG:


,story_id,rq2_validity,rq2_message,rq1_similarity
0,US002,1,Hợp lệ (Cú pháp chuẩn xác),0.791753
1,US006,1,Hợp lệ (Cú pháp chuẩn xác),0.759651
2,US007,1,Hợp lệ (Cú pháp chuẩn xác),0.793465
3,US008,1,Hợp lệ (Cú pháp chuẩn xác),0.759124
4,US009,1,Hợp lệ (Cú pháp chuẩn xác),0.912459
5,US015,1,Hợp lệ (Cú pháp chuẩn xác),0.905025
6,US016,1,Hợp lệ (Cú pháp chuẩn xác),0.883222
7,US018,1,Hợp lệ (Cú pháp chuẩn xác),0.820932
8,US035,1,Hợp lệ (Cú pháp chuẩn xác),0.858018
9,US041,1,Hợp lệ (Cú pháp chuẩn xác),0.777775



Đã xuất file kết quả: final_measurement_results.csv
